# Level 2: The Core Water Balance Model (Vectorized)

## 2.1 The Mathematical Objective
To simulate the daily moisture availability across the farm, we rely on the discrete soil-water balance model. The core equation governing this system is:

$$S_{t+1} = S_t + P_t + I_t - ET_t - D_t$$

**Where:**
* $S$: Soil Moisture (%)
* $P$: Precipitation / Rainfall (mm)
* $I$: Irrigation Applied (Control Variable, mm)
* $ET$: Evapotranspiration (System Loss, mm)
* $D$: Drainage (System Loss, mm)

## 2.2 Computational Approach
Because calculating this equation row-by-row using Python `for` loops introduces severe computational bottlenecks, this level implements **NumPy Vectorization**. By processing arrays at the C-language level, we can calculate $ET$ and $D$ for thousands of temporal records simultaneously.

In [ ]:
import pandas as pd
import numpy as np

# df_clean = pd.read_csv('../data/processed/cleaned_irrigation_dataset.csv')
# df_simulated = calculate_water_balance(df_clean)
# display(df_simulated.head())

def calculate_water_balance(df):
    """
    Executes vectorized water loss calculations (Evapotranspiration and Drainage)
    across the entire farm timeline simultaneously.
    """
    print("Initializing Vectorized Water Balance Engine...")




    # 1. Convert Pandas columns to fast NumPy arrays
    temps = df['avg_temp'].to_numpy()
    humidity = df['avg_humidity'].to_numpy()
    rain = df['total_rain'].to_numpy()
    moisture = df['soil_moisture_pct'].to_numpy()
    field_capacity = df['field_capacity_pct'].to_numpy()
    drainage_coeff = df['drainage_coefficient'].to_numpy()

    # 2. Vectorized Evapotranspiration (ET) Approximation
    # A localized approximation: Higher temperatures and lower humidity increase evaporation.
    et_raw = (temps * 0.4) - (humidity * 0.05)
    
    # We use np.clip to ensure physical boundaries (ET cannot be negative, 
    # and rarely exceeds 8.0mm/day in this climate zone)
    et_daily = np.clip(et_raw, a_min=0.5, a_max=8.0) 

    # 3. Vectorized Drainage (D)
    # Water only drains deep into the soil if the current moisture + new rain exceeds Field Capacity.
    excess_water = (moisture + rain) - field_capacity
    
    # np.where acts as a vectorized 'if' statement: 
    # If excess > 0, calculate drainage loss. Otherwise, 0 loss.
    drainage_daily = np.where(excess_water > 0, excess_water * drainage_coeff, 0.0)

    # 4. Calculate S_(t+1) - The Projected Next Day Moisture
    # S_{t+1} = S_t + P - ET - D (Holding Irrigation at 0 for baseline simulation)
    projected_moisture = moisture + rain - et_daily - drainage_daily

    # 5. Append calculated arrays back to the DataFrame
    df['estimated_ET_mm'] = np.round(et_daily, 2)
    df['estimated_drainage_mm'] = np.round(drainage_daily, 2)
    df['projected_moisture_pct'] = np.round(projected_moisture, 2)

    print("Vectorization Complete. Calculations applied at C-level speeds.")
    return df

# --- How to run this when your colleague finishes the data ---
# df_clean = pd.read_csv('../data/processed/cleaned_irrigation_dataset.csv')
# df_simulated = calculate_water_balance(df_clean)
# display(df_simulated.head())

## 2.4 Numerical Reliability & Error Analysis

In scientific computing, numbers are not absolute. Computers represent continuous real numbers using finite **floating-point arithmetic**, which inherently introduces rounding and truncation errors. 

### The Floating-Point Problem
For example, in standard IEEE 754 floating-point math, $0.1 + 0.2$ does not precisely equal $0.3$:

In [ ]:
# Demonstrating the classic floating point representation error
float_sum = 0.1 + 0.2
print(f"0.1 + 0.2 = {float_sum}")
print(f"Does 0.1 + 0.2 == 0.3? {float_sum == 0.3}")

# To handle this in scientific computing, we use np.isclose() instead of '=='
print(f"Using np.isclose(0.1 + 0.2, 0.3): {np.isclose(float_sum, 0.3)}")

### 2.5 Error Propagation Experiment
Beyond computer hardware limits, real-world IoT sensors have physical **measurement noise**. If our temperature sensor has an accuracy error of $\pm 0.5^\circ C$ and humidity has an error of $\pm 2.0\%$, how does that small hardware error propagate through our formulas and affect the final water loss calculation ($ET$)?

In [ ]:
import matplotlib.pyplot as plt

print("🧪 Running Error Propagation Experiment...")

# 1. Take a baseline subset of our data (e.g., just the Maize zone)
df_maize = df[df['crop_type'] == 'maize'].copy()

# 2. Extract our baseline (true) values
true_temp = df_maize['avg_temp'].to_numpy()
true_humidity = df_maize['avg_humidity'].to_numpy()

# Calculate 'True' ET (Without Noise)
true_et = np.clip((true_temp * 0.4) - (true_humidity * 0.05), 0.5, 8.0)

# 3. Simulate hardware Measurement Noise (Random Normal Distribution)
# Temperature sensor error: standard deviation of 0.5 degrees
# Humidity sensor error: standard deviation of 2.0 %
np.random.seed(42) # For reproducibility
noise_temp = np.random.normal(0, 0.5, len(true_temp))
noise_humidity = np.random.normal(0, 2.0, len(true_humidity))

# Apply noise to the sensors
noisy_temp = true_temp + noise_temp
noisy_humidity = true_humidity + noise_humidity

# Calculate 'Noisy' ET
noisy_et = np.clip((noisy_temp * 0.4) - (noisy_humidity * 0.05), 0.5, 8.0)

# Calculate the Absolute Error propagated into the final calculation
et_error = np.abs(noisy_et - true_et)
mean_et_error = np.mean(et_error)

print(f"Average ET Error introduced by sensor noise: ±{mean_et_error:.3f} mm/day")

# 4. Generate the required Error Propagation Plot
plt.figure(figsize=(12, 5))

# Plot True vs Noisy Data
plt.plot(df_maize['date'], true_et, label='True ET (Perfect Sensors)', color='blue', linewidth=2)
plt.plot(df_maize['date'], noisy_et, label='Noisy ET (Real-world Sensors)', color='red', linestyle='dashed', alpha=0.7)

# Fill the error gap to show the propagation
plt.fill_between(df_maize['date'], true_et, noisy_et, color='red', alpha=0.2, label='Propagated Error Margin')

plt.title('Error Propagation: How Sensor Noise Affects ET Calculations (Maize Zone)')
plt.ylabel('Evapotranspiration (mm/day)')
plt.xlabel('Date')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()

# Save the plot for the final report
plt.savefig('error_propagation_plot.png', dpi=300)
plt.show()